# Import des bibliothèques nécessaires

In [1]:
from pathlib import Path
import pandas as pd
import re



# Chargement des données

In [2]:
df=pd.read_csv(Path("..") /"data" /"2_processed"/ "01_paquets_phrases.csv", encoding="utf-8",)
df.head()

,nom_fichier,id_paquet,phrases_paquet
0,1893_20_Le_docteur_Pascal._clean.txt,0,Dans la chaleur de l’ardente après-midi de jui...
1,1893_20_Le_docteur_Pascal._clean.txt,1,"Dans sa longue blouse noire, elle était très g..."
2,1893_20_Le_docteur_Pascal._clean.txt,2,"Mais il dut prendre une chaise, la planche du ..."
3,1893_20_Le_docteur_Pascal._clean.txt,3,"Elle s’était redressée, le sang aux joues, les..."
4,1893_20_Le_docteur_Pascal._clean.txt,4,"Oh! Monsieur, la religion n’a jamais fait de m..."


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10638 entries, 0 to 10637
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   nom_fichier     10638 non-null  str  
 1   id_paquet       10638 non-null  int64
 2   phrases_paquet  10638 non-null  str  
dtypes: int64(1), str(2)
memory usage: 249.5 KB


# Préparation du dataframe 

In [4]:
df = df.rename(columns={
    "nom_fichier": "fichier",
    "id_paquet": "paquet_id",
    "phrases_paquet": "texte"
})

In [5]:
df["annee"] = df["fichier"].str.extract(r"^(\d{4})").astype(int)

In [6]:
ordre_romans= (
    df[["fichier", "annee"]]
    .drop_duplicates()
    .sort_values(["annee", "fichier"])
    .reset_index(drop=True)
)

ordre_romans["ordre_romans"] = range(1, len(ordre_romans) + 1)

df = df.merge(ordre_romans[["fichier", "ordre_romans"]], on="fichier", how="left")

In [7]:
df["roman"] = (
    df["fichier"]
    .str.replace(r"^\d{4}_\d+_", "", regex=True)
    .str.replace(r"_clean\.txt$", "", regex=True)
    .str.replace("_", " ")
)

In [8]:
df["nb_mots"] = df["texte"].str.split().str.len()

In [9]:
df = df[[
    "roman",
    "annee",
    "ordre_romans",
    "paquet_id",
    "texte",
    "nb_mots"
]]

In [10]:
df = df.sort_values(["ordre_romans", "paquet_id"]).reset_index(drop=True)

In [11]:
df["paquet_id"] = df.groupby("roman").cumcount() + 1

In [12]:
df.head()

,roman,annee,ordre_romans,paquet_id,texte,nb_mots
0,1865 La confession de Claude.,1865,1,1,"Voici l’hiver: l’air, au matin, devient plus f...",401
1,1865 La confession de Claude.,1865,1,2,Le grillon chantait; le souffle harmonieux des...,336
2,1865 La confession de Claude.,1865,1,3,"Pars cependant, puisque tu as soif de la vie. ...",248
3,1865 La confession de Claude.,1865,1,4,"Ai-je eu trop de confiance en ma force, ma pla...",336
4,1865 La confession de Claude.,1865,1,5,Le premier rayon a chassé le cauchemar de ma v...,359


In [13]:
chemin_sortie = Path("..") /"data" /"2_processed" /"02_corpus_zola.csv"
chemin_sortie.parent.mkdir(parents=True, exist_ok=True)

df.to_csv(chemin_sortie, index=False, encoding="utf-8")